# Task 1: Create a Prescription Parser using CRF
This task tests your ability to build a Doctor Prescription Parser with the help of CRF model

Your job is to build a Prescription Parser that takes a prescription (sentence) as an input and find / label the words in that sentence with one of the already pre-defined labels

### Problem: SEQUENCE PREDICTION - Label words in a sentence
#### Input : Doctor Prescription in the form of a sentence split into tokens
- Ex: Take 2 tablets once a day for 10 days

#### Output : FHIR Labels
- ('Take', 'Method')
- ('2', 'Qty')
- ('tablets', 'Form')
- ('once', 'Frequency')
- ('a', 'Period')
- ('day', 'PeriodUnit')
- ('for', 'FOR')
- ('10', 'Duration')
- ('days', 'DurationUnit')

### Major Steps
- Install necessary library
- Import the libraries
- Create training data with labels
    - Split the sentence into tokens
    - Compute POS tags
    - Create triples
- Extract features
- Split the data into training and testing set
- Create CRF model
- Save the CRF model
- Load the CRF model
- Predict on test data
- Accuracy

#### Install necesaary library

In [ ]:
pip install sklearn-crfsuite nltk

#### Import the necessary libraries

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger') # original line
nltk.download('averaged_perceptron_tagger_eng') # Download the specific English tagger data
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
import pickle
!pip install sklearn
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd



[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Input data (GIVEN)
#### Creating the inputs to the ML model in the following form:
- sigs --> ['take 3 tabs for 10 days']       INPUT SIG
- input_sigs --> [['take', '3', 'tabs', 'for', '10', 'days']]      TOKENS
- output_labels --> [['Method','Qty', 'Form', 'FOR', 'Duration', 'DurationUnit']]       LABELS

In [ ]:
sigs = ["for 5 to 6 days", "inject 2 units", "x 2 weeks", "x 3 days", "every day", "every 2 weeks", "every 3 days", "every 1 to 2 months", "every 2 to 6 weeks", "every 4 to 6 days", "take two to four tabs", "take 2 to 4 tabs", "take 3 tabs orally bid for 10 days at bedtime", "swallow three capsules tid orally", "take 2 capsules po every 6 hours", "take 2 tabs po for 10 days", "take 100 caps by mouth tid for 10 weeks", "take 2 tabs after an hour", "2 tabs every 4-6 hours", "every 4 to 6 hours", "q46h", "q4-6h", "2 hours before breakfast", "before 30 mins at bedtime", "30 mins before bed", "and 100 tabs twice a month", "100 tabs twice a month", "100 tabs once a month", "100 tabs thrice a month", "3 tabs daily for 3 days then 1 tab per day at bed", "30 tabs 10 days tid", "take 30 tabs for 10 days three times a day", "qid q6h", "bid", "qid", "30 tabs before dinner and bedtime", "30 tabs before dinner & bedtime", "take 3 tabs at bedtime", "30 tabs thrice daily for 10 days ", "30 tabs for 10 days three times a day", "Take 2 tablets a day", "qid for 10 days", "every day", "take 2 caps at bedtime", "apply 3 drops before bedtime", "take three capsules daily", "swallow 3 pills once a day", "swallow three pills thrice a day", "apply daily", "apply three drops before bedtime", "every 6 hours", "before food", "after food", "for 20 days", "for twenty days", "with meals"]
input_sigs = [['for', '5', 'to', '6', 'days'], ['inject', '2', 'units'], ['x', '2', 'weeks'], ['x', '3', 'days'], ['every', 'day'], ['every', '2', 'weeks'], ['every', '3', 'days'], ['every', '1', 'to', '2', 'months'], ['every', '2', 'to', '6', 'weeks'], ['every', '4', 'to', '6', 'days'], ['take', 'two', 'to', 'four', 'tabs'], ['take', '2', 'to', '4', 'tabs'], ['take', '3', 'tabs', 'orally', 'bid', 'for', '10', 'days', 'at', 'bedtime'], ['swallow', 'three', 'capsules', 'tid', 'orally'], ['take', '2', 'capsules', 'po', 'every', '6', 'hours'], ['take', '2', 'tabs', 'po', 'for', '10', 'days'], ['take', '100', 'caps', 'by', 'mouth', 'tid', 'for', '10', 'weeks'], ['take', '2', 'tabs', 'after', 'an', 'hour'], ['2', 'tabs', 'every', '4-6', 'hours'], ['every', '4', 'to', '6', 'hours'], ['q46h'], ['q4-6h'], ['2', 'hours', 'before', 'breakfast'], ['before', '30', 'mins', 'at', 'bedtime'], ['30', 'mins', 'before', 'bed'], ['and', '100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'once', 'a', 'month'], ['100', 'tabs', 'thrice', 'a', 'month'], ['3', 'tabs', 'daily', 'for', '3', 'days', 'then', '1', 'tab', 'per', 'day', 'at', 'bed'], ['30', 'tabs', '10', 'days', 'tid'], ['take', '30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['qid', 'q6h'], ['bid'], ['qid'], ['30', 'tabs', 'before', 'dinner', 'and', 'bedtime'], ['30', 'tabs', 'before', 'dinner', '&', 'bedtime'], ['take', '3', 'tabs', 'at', 'bedtime'], ['30', 'tabs', 'thrice', 'daily', 'for', '10', 'days'], ['30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['take', '2', 'tablets', 'a', 'day'], ['qid', 'for', '10', 'days'], ['every', 'day'], ['take', '2', 'caps', 'at', 'bedtime'], ['apply', '3', 'drops', 'before', 'bedtime'], ['take', 'three', 'capsules', 'daily'], ['swallow', '3', 'pills', 'once', 'a', 'day'], ['swallow', 'three', 'pills', 'thrice', 'a', 'day'], ['apply', 'daily'], ['apply', 'three', 'drops', 'before', 'bedtime'], ['every', '6', 'hours'], ['before', 'food'], ['after', 'food'], ['for', '20', 'days'], ['for', 'twenty', 'days'], ['with', 'meals']]
output_labels = [['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit'], ['Method', 'Qty', 'Form'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['EVERY', 'Period'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'Form', 'PO', 'BID', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'TID', 'PO'], ['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'PO', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'BY', 'PO', 'TID', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'AFTER', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Q46H'], ['Q4-6H'], ['Qty', 'PeriodUnit', 'BEFORE', 'WHEN'], ['BEFORE', 'Qty', 'M', 'AT', 'WHEN'], ['Qty', 'M', 'BEFORE', 'WHEN'], ['AND', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit', 'THEN', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'AT', 'WHEN'], ['Qty', 'Form', 'Duration', 'DurationUnit', 'TID'], ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Qty', 'TIMES', 'Period', 'PeriodUnit'], ['QID', 'Q6H'], ['BID'], ['QID'],['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Qty', 'Form', 'Frequency', 'DAILY', 'FOR', 'Duration', 'DurationUnit'], ['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Frequency', 'TIMES', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Period', 'PeriodUnit'], ['QID', 'FOR', 'Duration', 'DurationUnit'], ['EVERY', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['Method', 'Qty', 'Form', 'DAILY'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'DAILY'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['EVERY', 'Period', 'PeriodUnit'], ['BEFORE', 'FOOD'], ['AFTER', 'FOOD'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['WITH', 'FOOD']]

In [ ]:
len(sigs), len(input_sigs) , len(output_labels)

(56, 56, 56)

### Creating a Tuples Maker method
Create the tuples as given below by writing a function **tuples_maker(input_sigs, output_labels)** and returns **output** as given below

Input(s):
- input_sigs
- output_lables

Output:

[[('for', 'FOR'),
  ('5', 'Duration'),
  ('to', 'TO'),
  ('6', 'DurationMax'),
  ('days', 'DurationUnit')], [second sentence], ...]

In [ ]:
def tuples_maker(inp, out):

    return sample_data

In [ ]:
def tuples_maker(inp, out):
    """
    Creates a list of tuples, where each tuple contains a word from inp and its corresponding label from out.

    Args:
      inp: A list of lists, where each inner list contains the words of a sentence.
      out: A list of lists, where each inner list contains the labels for the words in the corresponding sentence in inp.

    Returns:
      A list of lists, where each inner list contains tuples of (word, label) for the corresponding sentence.
    """
    sample_data = []
    for sig, labels in zip(inp, out):
        tuples_list = [(word, label) for word, label in zip(sig, labels)]
        sample_data.append(tuples_list)
    return sample_data


### Creating the triples_maker( ) for feature extraction
- input: tuples_maker_output
- output:
[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')], [second sentence], ... ]

In [ ]:
def triples_maker(whole_data):
    """
    Adds POS tags to the tuples generated by tuples_maker.

    Args:
      whole_data: A list of lists, where each inner list contains tuples of (word, label) for a sentence.

    Returns:
      A list of lists, where each inner list contains tuples of (word, pos_tag, label) for a sentence.
    """
    sample_data = []
    for sentence_data in whole_data:
        words = [word for word, label in sentence_data]  # Extract words from sentence_data
        pos_tags = nltk.pos_tag(words)  # Get POS tags using nltk
        new_sentence_data = [(word, pos_tag, label)
                            for (word, label), (word_pos, pos_tag) in zip(sentence_data, pos_tags)]
        sample_data.append(new_sentence_data)
    return sample_data

# Calling tuples_maker to create whole_data
whole_data = tuples_maker(input_sigs, output_labels)

In [ ]:
sample_data = triples_maker(whole_data)
sample_data

[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')],
 [('inject', 'JJ', 'Method'), ('2', 'CD', 'Qty'), ('units', 'NNS', 'Form')],
 [('x', 'RB', 'FOR'),
  ('2', 'CD', 'Duration'),
  ('weeks', 'NNS', 'DurationUnit')],
 [('x', 'RB', 'FOR'),
  ('3', 'CD', 'Duration'),
  ('days', 'NNS', 'DurationUnit')],
 [('every', 'DT', 'EVERY'), ('day', 'NN', 'Period')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('3', 'CD', 'Period'),
  ('days', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('1', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('2', 'CD', 'PeriodMax'),
  ('months', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('4', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('

### Creating the features extractor method (GIVEN as a BASELINE)
#### The features used are:
- SOS, EOS, lowercase, uppercase, title, digit, postag, previous_tag, next_tag
#### Feel free to include more features

In [ ]:
def token_to_features(doc, i):
    word = doc[i][0]
    postag = doc[i][1]

    # Common features for all words
    features = [
        'bias',
        'word.lower=' + word.lower(),
        'word[-3:]=' + word[-3:],
        'word[-2:]=' + word[-2:],
        'word.isupper=%s' % word.isupper(),
        'word.istitle=%s' % word.istitle(),
        'word.isdigit=%s' % word.isdigit(),
        'postag=' + postag
    ]

    # Features for words that are not
    # at the beginning of a document
    if i > 0:
        word1 = doc[i-1][0]
        postag1 = doc[i-1][1]
        features.extend([
            '-1:word.lower=' + word1.lower(),
            '-1:word.istitle=%s' % word1.istitle(),
            '-1:word.isupper=%s' % word1.isupper(),
            '-1:word.isdigit=%s' % word1.isdigit(),
            '-1:postag=' + postag1
        ])
    else:
        # Indicate that it is the 'beginning of a document'
        features.append('BOS')

    # Features for words that are not
    # at the end of a document
    if i < len(doc)-1:
        word1 = doc[i+1][0]
        postag1 = doc[i+1][1]
        features.extend([
            '+1:word.lower=' + word1.lower(),
            '+1:word.istitle=%s' % word1.istitle(),
            '+1:word.isupper=%s' % word1.isupper(),
            '+1:word.isdigit=%s' % word1.isdigit(),
            '+1:postag=' + postag1
        ])
    else:
        # Indicate that it is the 'end of a document'
        features.append('EOS')

    return features

### Running the feature extractor on the training data
- Feature extraction
- Train-test-split

In [ ]:
# Feature extraction
X = [[token_to_features(doc, i) for i in range(len(doc))] for doc in sample_data]
y = [[label for word, postag, label in doc] for doc in sample_data]

# Train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Training the CRF model with the features extracted using the feature extractor method

In [ ]:

# Submit training data to the trainer
import sklearn_crfsuite
import pickle
# Create a CRF model
# Create and train the CRF model with verbose output
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
    verbose=True  # Enable verbose output
)

# Submit training data to the trainer (fit the model)
crf.fit(X_train, y_train)

# Set the parameters of the model
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
    verbose=True  # Enable verbose output
)
crf.fit(X_train, y_train)




# Providing a file name as a parameter to the train function, such that
# Save the trained model to a file
model_filename = 'crf_model.pkl'  # Specify your desired file name
with open(model_filename, 'wb') as file:
    pickle.dump(crf, file)

# the model will be saved to the file when training is finished


loading training data to CRFsuite: 100%|██████████| 44/44 [00:00<00:00, 12416.70it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 1763
Seconds required: 0.003

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=0.00  loss=568.14   active=1749  feature_norm=1.00
Iter 2   time=0.00  loss=387.36   active=1489  feature_norm=5.05
Iter 3   time=0.00  loss=282.30   active=1497  feature_norm=5.96
Iter 4   time=0.00  loss=238.76   active=1293  feature_norm=6.64
Iter 5   time=0.00  loss=173.39   active=1138  feature_norm=8.49
Iter 6   time=0.00  loss=118.89   active=1054  feature_norm=11.07
Iter 7   time=0.00  loss=90.36    active=1062  feature_norm=13.36
Iter 8   time=0.00  loss=81.34    active=1020  feature_norm=14.28
Iter 9   time=0.00  loss=77.74    active=1044  feature_norm=14.62
Iter 10  ti

loading training data to CRFsuite: 100%|██████████| 44/44 [00:00<00:00, 13338.35it/s]


Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 1763
Seconds required: 0.003

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=0.00  loss=568.14   active=1749  feature_norm=1.00
Iter 2   time=0.00  loss=387.36   active=1489  feature_norm=5.05
Iter 3   time=0.00  loss=282.30   active=1497  feature_norm=5.96
Iter 4   time=0.00  loss=238.76   active=1293  feature_norm=6.64
Iter 5   time=0.00  loss=173.39   active=1138  feature_norm=8.49
Iter 6   time=0.00  loss=118.89   active=1054  feature_norm=11.07
Iter 7   time=0.00  loss=90.36    active=1062  feature_norm=13.36
Iter 8   time=0.00  loss=81.34    active=1020  feature_norm=14.28
Iter 9   time=0.00  loss=77.74    active=1044  feature_norm=14.62
Iter 10  ti

### Predicting the test data with the built model

In [ ]:
import pickle

# Load the trained model
with open('crf_model.pkl', 'rb') as f:
    crf = pickle.load(f)

# Make predictions on the test data
y_pred = crf.predict(X_test)

# Print the predictions
print(y_pred)

[list(['FOR', 'Duration', 'TO', 'PeriodMax', 'PeriodUnit'])
 list(['EVERY', 'Period', 'PeriodUnit']) list(['QID'])
 list(['Method', 'Qty', 'Form', 'TID', 'DAILY'])
 list(['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'])
 list(['EVERY', 'Period', 'PeriodUnit'])
 list(['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'])
 list(['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'])
 list(['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'])
 list(['Method', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'])
 list(['FOR', 'Duration', 'DurationUnit'])
 list(['FOR', 'Duration', 'DurationUnit'])]


### Putting all the prediction logic inside a predict method

In [ ]:

   def predict(sig):
    """
    predict(sig)
    Purpose: Labels the given sig into corresponding labels
    @param sig. A Sentence  # A medical prescription sig written by a doctor
    @return     A list      # A list with predicted labels (first level of labeling)
    >>> predict('2 tabs every 4 hours')
    [['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit']]
    >>> predict('2 tabs with food')
    [['Qty', 'Form', 'WITH', 'FOOD']]
    >>> predict('2 tabs qid x 30 days')
    [['Qty', 'Form', 'QID', 'FOR', 'Duration', 'DurationUnit']]
    """
    # Tokenize the input sig
    tokens = nltk.word_tokenize(sig)

    # Create triples from tokens to get features for prediction
    triples = [(token, pos, 'NA') for token, pos in nltk.pos_tag(tokens)]

    # Extract features for each token
    features = [token_to_features(triples, i) for i in range(len(triples))]

    # Make predictions using the trained CRF model
    predictions = crf.predict([features])

    # updated section
    # Print the original input sentence
    print(sig)
    # Print the predictions in the desired format
    print(predictions)

    return predictions

### Sample predictions

In [ ]:
predictions = predict("take 2 tabs every 6 hours x 10 days")

take 2 tabs every 6 hours x 10 days
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'PeriodUnit' 'FOR' 'Duration'
  'DurationUnit']]


In [ ]:
predictions = predict("2 capsu for 10 day at bed")

2 capsu for 10 day at bed
[['Qty' 'Form' 'FOR' 'Duration' 'DurationUnit' 'AT' 'WHEN']]


In [ ]:
predictions = predict("2 capsu for 10 days at bed")

2 capsu for 10 days at bed
[['Qty' 'Form' 'FOR' 'Duration' 'DurationUnit' 'AT' 'WHEN']]


In [ ]:
predictions = predict("5 days 2 tabs at bed")

5 days 2 tabs at bed
[['Duration' 'DurationUnit' 'Qty' 'Form' 'AT' 'WHEN']]


In [ ]:
predictions = predict("3 tabs qid x 10 weeks")

3 tabs qid x 10 weeks
[['Qty' 'Form' 'QID' 'FOR' 'Duration' 'DurationUnit']]


In [ ]:
predictions = predict("x 30 days")

x 30 days
[['FOR' 'Duration' 'DurationUnit']]


In [ ]:
predictions = predict("x 20 months")

x 20 months
[['FOR' 'Duration' 'DurationUnit']]


In [ ]:
predictions = predict("take 2 tabs po tid for 10 days")

take 2 tabs po tid for 10 days
[['Method' 'Qty' 'Form' 'PO' 'TID' 'FOR' 'Duration' 'DurationUnit']]


In [ ]:
predictions = predict("take 2 capsules po every 6 hours")

take 2 capsules po every 6 hours
[['Method' 'Qty' 'Form' 'PO' 'EVERY' 'Period' 'PeriodUnit']]


In [ ]:
predictions = predict("inject 2 units pu tid")

inject 2 units pu tid
[['Method' 'Qty' 'Form' 'PO' 'TID']]


In [ ]:
predictions = predict("swallow 3 caps tid by mouth")

swallow 3 caps tid by mouth
[['Method' 'Qty' 'Form' 'TID' 'BY' 'PO']]


In [ ]:
predictions = predict("inject 3 units orally")

inject 3 units orally
[['Method' 'Qty' 'Form' 'Frequency']]


In [ ]:
predictions = predict("orally take 3 tabs tid")

orally take 3 tabs tid
[['Method' 'Method' 'Qty' 'Form' 'TID']]


In [ ]:
predictions = predict("by mouth take three caps")

by mouth take three caps
[['BY' 'PO' 'Method' 'Qty' 'Form']]


In [ ]:
predictions = predict("take 3 tabs orally three times a day for 10 days at bedtime")

take 3 tabs orally three times a day for 10 days at bedtime
[['Method' 'Qty' 'Form' 'Frequency' 'Qty' 'TIMES' 'Period' 'PeriodUnit'
  'FOR' 'Duration' 'DurationUnit' 'AT' 'WHEN']]


In [ ]:
predictions = predict("take 3 tabs orally bid for 10 days at bedtime")

take 3 tabs orally bid for 10 days at bedtime
[['Method' 'Qty' 'Form' 'Frequency' 'PeriodUnit' 'FOR' 'Duration'
  'DurationUnit' 'AT' 'WHEN']]


In [ ]:
predictions = predict("take 3 tabs bid orally at bed")

take 3 tabs bid orally at bed
[['Method' 'Qty' 'Form' 'Frequency' 'PeriodUnit' 'AT' 'WHEN']]


In [ ]:
predictions = predict("take 10 capsules by mouth qid")

take 10 capsules by mouth qid
[['Method' 'Qty' 'Form' 'BY' 'PO' 'QID']]


In [ ]:
predictions = predict("inject 10 units orally qid x 3 months")

inject 10 units orally qid x 3 months
[['Method' 'Qty' 'Form' 'Frequency' 'QID' 'FOR' 'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("please take 2 tablets per day for a month in the morning and evening each day")

please take 2 tablets per day for a month in the morning and evening each day
[['Method' 'Method' 'Qty' 'Form' 'Frequency' 'PeriodUnit' 'FOR' 'Period'
  'PeriodUnit' 'EVERY' 'Period' 'PeriodUnit' 'AND' 'EVERY' 'Period'
  'PeriodUnit']]


In [ ]:
prediction = predict("Amoxcicillin QID 30 tablets")

Amoxcicillin QID 30 tablets
[['Method' 'Method' 'Qty' 'Form']]


In [ ]:
prediction = predict("take 3 tabs TID for 90 days with food")

take 3 tabs TID for 90 days with food
[['Method' 'Qty' 'Form' 'Frequency' 'FOR' 'Duration' 'DurationUnit'
  'WITH' 'FOOD']]


In [ ]:
prediction = predict("with food take 3 tablets per day for 90 days")

with food take 3 tablets per day for 90 days
[['WITH' 'FOOD' 'Method' 'Qty' 'Form' 'Frequency' 'PeriodUnit' 'FOR'
  'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("with food take 3 tablets per week for 90 weeks")

with food take 3 tablets per week for 90 weeks
[['WITH' 'FOOD' 'Method' 'Qty' 'Form' 'Frequency' 'PeriodUnit' 'FOR'
  'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("take 2-4 tabs")

take 2-4 tabs
[['Method' 'Qty' 'Form']]


In [ ]:
prediction = predict("take 2 to 4 tabs")

take 2 to 4 tabs
[['Method' 'Qty' 'TO' 'Qty' 'Form']]


In [ ]:
prediction = predict("take two to four tabs")

take two to four tabs
[['Method' 'Qty' 'TO' 'Qty' 'Form']]


In [ ]:
prediction = predict("take 2-4 tabs for 8 to 9 days")

take 2-4 tabs for 8 to 9 days
[['Method' 'Qty' 'Form' 'FOR' 'Duration' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("take 20 tabs every 6 to 8 days")

take 20 tabs every 6 to 8 days
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("take 2 tabs every 4 to 6 days")

take 2 tabs every 4 to 6 days
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("take 2 tabs every 2 to 10 weeks")

take 2 tabs every 2 to 10 weeks
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("take 2 tabs every 4 to 6 days")

take 2 tabs every 4 to 6 days
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("take 2 tabs every 2 to 10 months")

take 2 tabs every 2 to 10 months
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("every 60 mins")

every 60 mins
[['EVERY' 'Period' 'PeriodUnit']]


In [ ]:
prediction = predict("every 10 mins")

every 10 mins
[['EVERY' 'Period' 'PeriodUnit']]


In [ ]:
prediction = predict("every two to four months")

every two to four months
[['EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("take 2 tabs every 3 to 4 days")

take 2 tabs every 3 to 4 days
[['Method' 'Qty' 'Form' 'EVERY' 'Period' 'TO' 'PeriodMax' 'PeriodUnit']]


In [ ]:
prediction = predict("every 3 to 4 days take 20 tabs")

every 3 to 4 days take 20 tabs
[['EVERY' 'Period' 'TO' 'Duration' 'DurationUnit' 'Method' 'Qty' 'Form']]


In [ ]:
prediction = predict("once in every 3 days take 3 tabs")

once in every 3 days take 3 tabs
[['Frequency' 'Period' 'EVERY' 'Period' 'PeriodUnit' 'Method' 'Qty'
  'Form']]


In [ ]:
prediction = predict("take 3 tabs once in every 3 days")

take 3 tabs once in every 3 days
[['Method' 'Qty' 'Form' 'Frequency' 'Period' 'EVERY' 'Period'
  'PeriodUnit']]


In [ ]:
prediction = predict("orally take 20 tabs every 4-6 weeks")

orally take 20 tabs every 4-6 weeks
[['Method' 'Method' 'Qty' 'Form' 'EVERY' 'Period' 'PeriodUnit']]


In [ ]:
prediction = predict("10 tabs x 2 days")

10 tabs x 2 days
[['Qty' 'Form' 'FOR' 'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("3 capsule x 15 days")

3 capsule x 15 days
[['Qty' 'Form' 'FOR' 'Duration' 'DurationUnit']]


In [ ]:
prediction = predict("10 tabs")

10 tabs
[['Qty' 'Form']]


#Evaluation: Classification Report and Confusion Matrix

In [ ]:
# Flatten the predictions and true labels
y_pred_flat = [item for sublist in y_pred for item in sublist]
y_test_flat = [item for sublist in y_test for item in sublist]

# Generate and print the classification report
print(classification_report(y_test_flat, y_pred_flat))

# Generate and print the confusion matrix
conf_matrix = confusion_matrix(y_test_flat, y_pred_flat)

# Display confusion matrix using pandas for better formatting
labels = sorted(list(set(y_test_flat + y_pred_flat)))  # Get unique labels
df_cm = pd.DataFrame(conf_matrix, index=labels, columns=labels)
print("\nConfusion Matrix:")
display(df_cm) # Use display for a formatted output in Jupyter Notebook

              precision    recall  f1-score   support

         AND       1.00      1.00      1.00         1
          AT       1.00      1.00      1.00         1
      BEFORE       1.00      1.00      1.00         2
         BID       0.00      0.00      0.00         2
       DAILY       0.00      0.00      0.00         0
    Duration       1.00      1.00      1.00         4
 DurationMax       0.00      0.00      0.00         1
DurationUnit       1.00      0.75      0.86         4
       EVERY       1.00      1.00      1.00         3
         FOR       1.00      1.00      1.00         4
        Form       1.00      1.00      1.00         5
   Frequency       0.50      1.00      0.67         1
      Method       1.00      1.00      1.00         3
          PO       0.00      0.00      0.00         2
      Period       1.00      1.00      1.00         4
   PeriodMax       0.50      1.00      0.67         1
  PeriodUnit       0.67      1.00      0.80         4
         QID       0.00    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

,AND,AT,BEFORE,BID,DAILY,Duration,DurationMax,DurationUnit,EVERY,FOR,...,Method,PO,Period,PeriodMax,PeriodUnit,QID,Qty,TID,TO,WHEN
AND,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AT,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BEFORE,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BID,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
DAILY,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Duration,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DurationMax,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
DurationUnit,0,0,0,0,0,0,0,3,0,0,...,0,0,0,0,1,0,0,0,0,0
EVERY,0,0,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
FOR,0,0,0,0,0,0,0,0,0,4,...,0,0,0,0,0,0,0,0,0,0


# My observation:

If we see high precision but low recall for a certain label, it means that when our model does predict that label, it's usually correct, but it might be missing a lot of actual instances of that label.
If we notice many false positives for a label in the confusion matrix, it could indicate that the model is overly sensitive to features related to that label.
By comparing different rows and columns in the confusion matrix, we can gain insights into specific areas of confusion for the model (e.g., maybe it's misclassifying "DurationUnit" and "PeriodUnit" often).
